# 03 · Deep Researcher Agent

**Goal:** Take the ranked `GapReport` from the Gap Analysis agent and produce **personalized learning pathways** — courses, projects, checklists — grounded in current web knowledge via Tavily search.

This notebook implements the architecture from `Documentation/Deep Researcher Agentic System.pdf`:

```
        ┌────────────────── Deep Researcher LangGraph ──────────────────┐
Gap ───▶│  Researcher (plans + critiques) ⇄  Tavily Search (online)     │
Report  │                       │                                       │
        │                       ▼                                       │
        │            Structuring Agent (recursive Pydantic)             │
        └────────────────────────┬──────────────────────────────────────┘
                                 ▼
                        Learning Pathways
```

### Key design decisions
- **Researcher ↔ Tavily loop** with a `continue | stop` router. Budget is measured in search iterations (default 3) AND in "gathered notes" minimum.
- **Critic step inside the researcher.** After each Tavily call the researcher decides: (a) do I have enough, or (b) what's missing — and drafts the next query. This is what separates a "search once and summarize" pipeline from a deep researcher.
- **Structuring agent** uses a **recursive Pydantic model** (`Pathway` contains `Milestone` which contains `Resource`) so Gemini emits a tree in one call with automatic validation.
- Entire thing lives in a LangGraph subgraph with shared state — drops straight into the supervisor.


## 0. Setup

In [ ]:
# !pip install -q langchain langchain-google-genai langchain-groq langchain-community tavily-python langgraph pydantic python-dotenv

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(Path("../backend/.env").resolve())
assert os.getenv("GOOGLE_API_KEY")
assert os.getenv("TAVILY_API_KEY")
print("OK")

## 1. Tools — Tavily search

`TavilySearchResults` returns `[{url, content, ...}]`. We'll keep `search_depth="advanced"` for research-grade results and `max_results=5` to stay inside free-tier budgets.


In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily = TavilySearchResults(
    max_results=5,
    search_depth="advanced",
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
)

# Smoke test
sample = tavily.invoke({"query": "best course to learn PyTorch in 2025"})
print(f"{len(sample)} results")
print(sample[0]["url"], "—", sample[0]["content"][:120])

## 2. Output schema — recursive Pydantic model

A `Pathway` is a tree: `Pathway → Milestone[] → Resource[]`. Gemini emits this in one structured call at the end of the research loop.


In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field

class Resource(BaseModel):
    title: str
    kind: Literal["course", "doc", "video", "article", "project"]
    provider: str = Field(description="e.g. Coursera, YouTube, official docs, fast.ai")
    url: str
    why: str = Field(description="1 sentence — why this resource for this milestone")

class Milestone(BaseModel):
    phase: Literal["Foundations", "Intermediate", "Advanced"]
    skill: str = Field(description="Gap skill being closed")
    estimated_weeks: int
    objective: str = Field(description="What the learner will be able to do after this milestone")
    resources: List[Resource] = Field(default_factory=list)
    checklist: List[str] = Field(description="3-5 concrete, checkable items")
    mini_project: Optional[str] = Field(None, description="Hands-on project suggestion")

class Pathway(BaseModel):
    target_role: str
    milestones: List[Milestone]
    rationale: str = Field(description="2-3 sentence reasoning on why the ordering works")

## 3. Researcher state + nodes

State fields:
- `gaps`: the incoming `GapReport.gaps` list
- `target_role`
- `notes`: list[str] — accumulated search findings, one entry per round
- `last_query`: the query just executed (for logging)
- `iteration`: round counter
- `max_iter`: hard stop
- `pathway`: final output

Nodes:
- `planner` — given gaps + notes so far, emit the next search query. On iteration 0 the query is broad; later iterations drill into whatever's still missing.
- `search` — call Tavily with `last_query`, append trimmed results to `notes`.
- `critic` — decide "continue" (need more notes) or "structure" (enough).
- `structurer` — one Gemini call that emits the full `Pathway`.


In [ ]:
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel as PydBase

gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)

# Minimal shape of an incoming gap (mirrors notebook 02's Gap).
class GapIn(PydBase):
    skill: str
    category: str
    relevance: int
    difficulty: str
    prerequisites: List[str]
    why: str

class ResearcherState(TypedDict, total=False):
    gaps: List[GapIn]
    target_role: str
    notes: List[str]
    last_query: str
    iteration: int
    max_iter: int
    pathway: Pathway

### Planner

The planner produces one query per iteration. Prompt it to (a) cover the highest-relevance gap first, (b) drill into gaps the previous searches didn't cover.


In [ ]:
class NextQuery(PydBase):
    query: str = Field(description="The single best next search query — concise, 6-15 words")
    rationale: str = Field(description="Why this query, 1 sentence")

plan_prompt = ChatPromptTemplate.from_template('''You are a research planner for career-skill learning pathways.

TARGET ROLE: {target_role}

OUTSTANDING GAPS (skill, relevance, difficulty):
{gaps}

NOTES GATHERED SO FAR (from previous searches, newest last):
{notes}

Propose ONE next web search query that will most improve the pathway.
Rules:
- Iteration 0: start with the highest-relevance gap and look for current (2025+) top courses/docs.
- Later iterations: cover gaps not yet represented in the notes, or drill deeper on a specific resource.
- Prefer concrete queries ("best PyTorch course 2025 for production ML") over vague ones.''')

planner = plan_prompt | gemini.with_structured_output(NextQuery)

def node_plan(state: ResearcherState) -> ResearcherState:
    gaps_str = "\n".join(f"- {g.skill} (rel={g.relevance}, {g.difficulty})" for g in state["gaps"])
    notes_str = "\n".join(f"[{i}] {n[:300]}" for i, n in enumerate(state.get("notes", []))) or "(none yet)"
    nxt: NextQuery = planner.invoke({
        "target_role": state["target_role"],
        "gaps": gaps_str,
        "notes": notes_str,
    })
    print(f"iter {state.get('iteration', 0)} → plan: {nxt.query}  ({nxt.rationale})")
    return {"last_query": nxt.query, "iteration": state.get("iteration", 0) + 1}

### Search node

In [ ]:
def node_search(state: ResearcherState) -> ResearcherState:
    results = tavily.invoke({"query": state["last_query"]})
    # Compress each result into a single line: url + first 500 chars.
    summary = "\n".join(f"{r['url']} :: {r.get('content','')[:500]}" for r in results)
    notes = state.get("notes", []) + [f"Q={state['last_query']}\n{summary}"]
    return {"notes": notes}

### Critic / router

Given the notes gathered, do we have enough? The router is a simple LLM classifier. We also hard-cap at `max_iter` so Tavily doesn't burn quota on a degenerate loop.


In [ ]:
class CriticOut(PydBase):
    decision: Literal["continue", "structure"]
    rationale: str

critic_prompt = ChatPromptTemplate.from_template('''You are a research critic.

TARGET ROLE: {target_role}
GAPS TO COVER:
{gaps}

NOTES GATHERED (newest last):
{notes}

Decide:
- "continue" if the notes don't yet give us enough to recommend concrete resources for EVERY gap.
- "structure" if we can now write a high-quality learning pathway with real links for every gap.''')

critic = critic_prompt | gemini.with_structured_output(CriticOut)

def node_critic_route(state: ResearcherState) -> str:
    # Hard stop on max iterations.
    if state.get("iteration", 0) >= state.get("max_iter", 3):
        print("→ max_iter reached, forcing structure")
        return "structure"
    gaps_str = "\n".join(f"- {g.skill}" for g in state["gaps"])
    notes_str = "\n".join(f"[{i}] {n[:500]}" for i, n in enumerate(state.get("notes", [])))
    out: CriticOut = critic.invoke({
        "target_role": state["target_role"],
        "gaps": gaps_str,
        "notes": notes_str,
    })
    print(f"→ critic: {out.decision} — {out.rationale}")
    return out.decision

### Structurer

One call. Takes gaps + all notes and emits the full `Pathway` tree.


In [ ]:
struct_prompt = ChatPromptTemplate.from_template('''You are a senior learning-path designer.

TARGET ROLE: {target_role}

USER'S SKILL GAPS (ordered by relevance, prerequisites first):
{gaps}

RESEARCH NOTES (URLs + extracts from authoritative sources):
{notes}

Produce a Pathway with:
- ONE milestone per gap, in prerequisite-respecting order.
- Each milestone in an appropriate phase (Foundations / Intermediate / Advanced).
- 2-4 real resources per milestone, PREFERRING URLs that appear in the research notes.
- A 3-5 item `checklist` of concrete actions.
- A `mini_project` that exercises the skill in context of the target role.
- `rationale`: why this ordering works for this user.''')

structurer = struct_prompt | gemini.with_structured_output(Pathway)

def node_structure(state: ResearcherState) -> ResearcherState:
    gaps_str = "\n".join(
        f"- {g.skill} (rel={g.relevance}, {g.difficulty}, prereqs={g.prerequisites})" for g in state["gaps"]
    )
    notes_str = "\n".join(state.get("notes", []))
    pathway: Pathway = structurer.invoke({
        "target_role": state["target_role"],
        "gaps": gaps_str,
        "notes": notes_str,
    })
    return {"pathway": pathway}

## 4. Wire up the LangGraph

In [ ]:
from langgraph.graph import StateGraph, START, END

g = StateGraph(ResearcherState)
g.add_node("plan",      node_plan)
g.add_node("search",    node_search)
g.add_node("structure", node_structure)

g.add_edge(START, "plan")
g.add_edge("plan", "search")
g.add_conditional_edges(
    "search",
    node_critic_route,            # returns "continue" or "structure"
    {"continue": "plan", "structure": "structure"},
)
g.add_edge("structure", END)

researcher = g.compile()

try:
    from IPython.display import Image, display
    display(Image(researcher.get_graph().draw_mermaid_png()))
except Exception:
    print(researcher.get_graph().draw_mermaid())

## 5. Run it end-to-end

Feed in sample gaps (would come from notebook 02) and see the loop do plan → search → [critic decides] → plan → search → ... → structure.


In [ ]:
# Sample gaps (normally produced by 02_gap_analysis_rag.ipynb)
sample_gaps = [
    GapIn(skill="Transformers",      category="ML",  relevance=95, difficulty="Hard",
          prerequisites=["PyTorch"],
          why="Core architecture for modern NLP/LLM work; expected of every ML engineer."),
    GapIn(skill="PyTorch",           category="ML",  relevance=92, difficulty="Medium",
          prerequisites=["Python", "Linear Algebra"],
          why="Primary deep learning framework in production ML roles."),
    GapIn(skill="MLflow",            category="MLOps", relevance=75, difficulty="Easy",
          prerequisites=["Python"],
          why="Standard for experiment tracking + model registry."),
]

state_in = {
    "gaps": sample_gaps,
    "target_role": "ml-engineer",
    "notes": [],
    "iteration": 0,
    "max_iter": 3,  # keep this small for Tavily quota
}

final = researcher.invoke(state_in, {"recursion_limit": 20})
path: Pathway = final["pathway"]

In [ ]:
print(f"Target role: {path.target_role}")
print(f"Rationale:   {path.rationale}\n")
for i, m in enumerate(path.milestones, 1):
    print(f"{i}. [{m.phase:13s}] {m.skill}  ({m.estimated_weeks}w)")
    print(f"   objective: {m.objective}")
    for r in m.resources:
        print(f"     · [{r.kind}] {r.title} — {r.provider}")
        print(f"       {r.url}")
    print(f"   checklist: {m.checklist}")
    print(f"   project:   {m.mini_project}\n")

## 6. Notes for the next engineer

- **Iteration budget.** `max_iter=3` for demos. Bump to 5-6 in production; keep the critic honest so most queries terminate early on simple profiles.
- **Tavily cost control.** Each loop spends 1 Tavily call. With 5 results × 3 iterations you typically hit ~15 URLs in the notes. Cache per (target_role, gaps_hash) in Redis.
- **Note compression.** Right now we truncate each result at 500 chars. If you swap in larger context, consider an additional summarizer node between `search` and `critic` that distills notes into bullet points — keeps the structurer prompt short.
- **Grounding.** Tell the structurer to prefer URLs that appear in notes (already in the prompt). In production add a post-hoc validator: for every emitted URL, assert it was in the notes. If not, either drop the resource or flag it.
- **Critic drift.** In long runs the critic can ping-pong on "continue". Safeguard: if the same query is proposed twice in a row, force "structure".
- **Downstream.** The `Pathway` is either shipped directly to the frontend or persisted to Supabase as `milestones` rows (see `sql/001_schema.sql`).


## 7. Evaluation — LLM as a Judge (Groq)

This section adds a **model-based evaluator** for pathway quality. We score each pathway on a rubric and return structured feedback.

### What is judged
- **Coverage**: Are all gaps addressed with concrete milestones?
- **Ordering**: Do milestones respect prerequisite logic?
- **Resource quality**: Are links plausible, relevant, and role-aligned?
- **Actionability**: Are checklist items and projects concrete and doable?
- **Personalization**: Does the plan reflect target role + user context?
- **Grounding**: Does the pathway use evidence from notes (not hallucinated claims)?

We use a strict JSON schema output so this can later plug into backend tests/CI.

In [ ]:
from langchain_groq import ChatGroq
from statistics import mean
from datetime import datetime, timezone

# You said you'll provide this key; set it in backend/.env before running.
assert os.getenv("GROQ_API_KEY"), "Missing GROQ_API_KEY in backend/.env"

judge_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.0,
)

class JudgeScore(PydBase):
    criterion: Literal[
        "coverage",
        "ordering",
        "resource_quality",
        "actionability",
        "personalization",
        "grounding",
    ]
    score: int = Field(ge=1, le=5)
    rationale: str = Field(description="One concise reason with evidence")

class JudgeVerdict(PydBase):
    overall_score: float = Field(ge=1, le=5)
    pass_fail: Literal["pass", "fail"]
    strengths: List[str]
    weaknesses: List[str]
    improvement_actions: List[str] = Field(description="Concrete fixes to apply to prompts/nodes")
    rubric_scores: List[JudgeScore]
    confidence: Literal["low", "medium", "high"]

judge_prompt = ChatPromptTemplate.from_template('''You are a strict evaluator for AI-generated career learning pathways.

Evaluate the pathway against this rubric (1=poor, 5=excellent):
1) coverage
2) ordering
3) resource_quality
4) actionability
5) personalization
6) grounding

PASS RULE:
- pass only if overall_score >= 4.0 AND no criterion is below 3.

INPUT
TARGET ROLE: {target_role}
USER GAPS:
{gaps}
RESEARCH NOTES:
{notes}
PATHWAY JSON:
{pathway_json}

Return a JudgeVerdict object only.''')

judge = judge_prompt | judge_llm.with_structured_output(JudgeVerdict)
print("Judge ready")

In [ ]:
def _gaps_to_text(gaps: List[GapIn]) -> str:
    return "\n".join(
        f"- {g.skill} (rel={g.relevance}, diff={g.difficulty}, prereqs={g.prerequisites}, why={g.why})"
        for g in gaps
    )


def evaluate_pathway_with_llm_judge(
    pathway: Pathway,
    gaps: List[GapIn],
    target_role: str,
    notes: List[str],
) -> JudgeVerdict:
    payload = {
        "target_role": target_role,
        "gaps": _gaps_to_text(gaps),
        "notes": "\n".join(notes)[:12000] if notes else "(no notes provided)",
        "pathway_json": pathway.model_dump_json(indent=2),
    }
    return judge.invoke(payload)


def compact_verdict(verdict: JudgeVerdict) -> dict:
    by_criterion = {s.criterion: s.score for s in verdict.rubric_scores}
    return {
        "overall_score": verdict.overall_score,
        "pass_fail": verdict.pass_fail,
        "confidence": verdict.confidence,
        "criterion_scores": by_criterion,
        "weaknesses": verdict.weaknesses,
        "improvement_actions": verdict.improvement_actions,
    }

In [ ]:
# Evaluate the pathway generated in Section 5
verdict = evaluate_pathway_with_llm_judge(
    pathway=path,
    gaps=sample_gaps,
    target_role=state_in["target_role"],
    notes=final.get("notes", []),
)

summary = compact_verdict(verdict)
print(summary)

### Optional: A/B eval loop for prompt tuning

Use this when experimenting with planner/structurer prompts. It runs the graph multiple times and reports judge scores so you can compare variants.

In [ ]:
def run_judged_trial(state_seed: dict, run_name: str = "trial") -> dict:
    run_out = researcher.invoke(state_seed, {"recursion_limit": 20})
    run_path: Pathway = run_out["pathway"]
    run_verdict = evaluate_pathway_with_llm_judge(
        pathway=run_path,
        gaps=state_seed["gaps"],
        target_role=state_seed["target_role"],
        notes=run_out.get("notes", []),
    )
    return {
        "run_name": run_name,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "overall": run_verdict.overall_score,
        "pass_fail": run_verdict.pass_fail,
        "confidence": run_verdict.confidence,
        "criterion_scores": {s.criterion: s.score for s in run_verdict.rubric_scores},
        "actions": run_verdict.improvement_actions,
    }


trials = [
    run_judged_trial({**state_in, "max_iter": 2}, run_name="short_budget"),
    run_judged_trial({**state_in, "max_iter": 4}, run_name="larger_budget"),
]

for t in trials:
    print("=" * 72)
    print(f"{t['run_name']} | overall={t['overall']:.2f} | {t['pass_fail']} | conf={t['confidence']}")
    print("criterion:", t["criterion_scores"])
    print("actions:")
    for a in t["actions"]:
        print(" -", a)

print("=" * 72)
print("avg overall:", round(mean(t["overall"] for t in trials), 3))

## 8. Trajectory Testing — Did the graph take the right path?

For probabilistic systems, output quality is necessary but not sufficient. We also validate the **execution trajectory** of the LangGraph.

This section adds:
- **Path tracing**: capture executed node sequence (`plan/search/structure`).
- **Invariant checks**: required transitions and budget constraints.
- **Drift detection**: compare current path vs golden baseline paths.
- **Optional LLM route judge**: qualitative assessment of route correctness/risk.

In [ ]:
import json
from pathlib import Path as FsPath

ALLOWED_TRANSITIONS = {
    ("plan", "search"),
    ("search", "plan"),
    ("search", "structure"),
}


def collect_node_path(app, state_seed: dict, recursion_limit: int = 20) -> tuple[list[str], dict]:
    """Collects executed node names in order by streaming graph updates."""
    path = []
    final_delta = {}
    for event in app.stream(state_seed, {"recursion_limit": recursion_limit}):
        if not isinstance(event, dict):
            continue
        for node_name, delta in event.items():
            # Skip internal terminal marker if present.
            if node_name.startswith("__"):
                continue
            path.append(node_name)
            if isinstance(delta, dict):
                final_delta.update(delta)
    return path, final_delta


def path_invariants(path: list[str], max_iter: int) -> dict:
    transitions = list(zip(path, path[1:]))
    bad_transitions = [t for t in transitions if t not in ALLOWED_TRANSITIONS]

    checks = {
        "starts_with_plan": bool(path) and path[0] == "plan",
        "ends_with_structure": bool(path) and path[-1] == "structure",
        "contains_structure": "structure" in path,
        "search_within_budget": path.count("search") <= max_iter,
        "no_illegal_transition": len(bad_transitions) == 0,
    }

    score = sum(1 for v in checks.values() if v) / len(checks)

    return {
        "checks": checks,
        "pass": all(checks.values()),
        "score": round(score, 3),
        "path": path,
        "search_calls": path.count("search"),
        "bad_transitions": bad_transitions,
    }


def levenshtein(a: list[str], b: list[str]) -> int:
    """Simple DP edit distance for path drift measurement."""
    if not a:
        return len(b)
    if not b:
        return len(a)
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        dp[i][0] = i
    for j in range(len(b) + 1):
        dp[0][j] = j

    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost,
            )
    return dp[-1][-1]


def normalized_drift(observed: list[str], expected: list[str]) -> float:
    if not expected and not observed:
        return 0.0
    denom = max(len(expected), len(observed), 1)
    return round(levenshtein(observed, expected) / denom, 3)


BASELINE_PATH = FsPath("Notebooks/.eval/deep_research_path_baseline.json")


def save_path_baseline(cases: list[dict], file_path: FsPath = BASELINE_PATH) -> dict:
    """Run once on a stable prompt version to create a golden route baseline."""
    rows = []
    for c in cases:
        observed_path, _ = collect_node_path(researcher, c["state"])
        rows.append(
            {
                "case_id": c["case_id"],
                "expected_path": observed_path,
                "max_iter": c["state"]["max_iter"],
            }
        )

    payload = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "rows": rows,
    }
    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return payload


def load_path_baseline(file_path: FsPath = BASELINE_PATH) -> dict:
    return json.loads(file_path.read_text(encoding="utf-8"))


def evaluate_path_drift(cases: list[dict], baseline: dict) -> list[dict]:
    baseline_by_id = {r["case_id"]: r for r in baseline["rows"]}
    report = []

    for c in cases:
        case_id = c["case_id"]
        observed_path, _ = collect_node_path(researcher, c["state"])
        inv = path_invariants(observed_path, c["state"]["max_iter"])

        expected = baseline_by_id[case_id]["expected_path"]
        drift = normalized_drift(observed_path, expected)

        report.append(
            {
                "case_id": case_id,
                "pass": inv["pass"],
                "invariant_score": inv["score"],
                "drift": drift,
                "expected": expected,
                "observed": observed_path,
                "bad_transitions": inv["bad_transitions"],
            }
        )
    return report

In [ ]:
# Define a tiny fixed input suite for trajectory tests.
path_test_cases = [
    {
        "case_id": "ml_engineer_core",
        "state": {
            "gaps": sample_gaps,
            "target_role": "ml-engineer",
            "notes": [],
            "iteration": 0,
            "max_iter": 3,
        },
    },
    {
        "case_id": "ml_engineer_low_budget",
        "state": {
            "gaps": sample_gaps,
            "target_role": "ml-engineer",
            "notes": [],
            "iteration": 0,
            "max_iter": 1,
        },
    },
]

# One-time baseline capture (uncomment once after prompt architecture is stable):
# baseline_payload = save_path_baseline(path_test_cases)
# print("saved baseline at", BASELINE_PATH)

print("cases ready:", [c["case_id"] for c in path_test_cases])

In [ ]:
# Run drift check against previously saved baseline.
baseline = load_path_baseline()
drift_report = evaluate_path_drift(path_test_cases, baseline)

for row in drift_report:
    print("=" * 88)
    print(f"case={row['case_id']} | pass={row['pass']} | invariant_score={row['invariant_score']} | drift={row['drift']}")
    print("expected:", row["expected"])
    print("observed:", row["observed"])
    if row["bad_transitions"]:
        print("bad transitions:", row["bad_transitions"])

avg_drift = round(mean(r["drift"] for r in drift_report), 3)
print("=" * 88)
print("avg drift:", avg_drift)
print("route status:", "OK" if avg_drift <= 0.2 and all(r["pass"] for r in drift_report) else "DRIFT ALERT")

In [ ]:
class RouteJudgeVerdict(PydBase):
    route_quality: Literal["good", "borderline", "bad"]
    drift_risk: Literal["low", "medium", "high"]
    must_fix: List[str]
    evidence: List[str]


route_judge_prompt = ChatPromptTemplate.from_template('''You are judging whether a LangGraph trajectory indicates agent drift.

EXPECTED PATH:
{expected_path}

OBSERVED PATH:
{observed_path}

INVARIANT SUMMARY:
{invariants}

Rules:
- "good" only if observed path is semantically equivalent to expected path and no critical invariant failures.
- "borderline" if small deviation but likely still safe.
- "bad" if deviations suggest routing/control drift.

Return RouteJudgeVerdict only.''')

route_judge = route_judge_prompt | judge_llm.with_structured_output(RouteJudgeVerdict)

# Example: judge the first drift row.
example = drift_report[0]
route_verdict = route_judge.invoke(
    {
        "expected_path": example["expected"],
        "observed_path": example["observed"],
        "invariants": {
            "pass": example["pass"],
            "score": example["invariant_score"],
            "bad_transitions": example["bad_transitions"],
            "drift": example["drift"],
        },
    }
)
print(route_verdict.model_dump())